# Spotify Personal Music Intelligence
## Notebook 02 — Feature Engineering

In this notebook I will take the cleaned Spotify listening data and create useful features for analysis and machine learning.

### What is feature engineering?

Feature engineering means creating useful measurements from existing data.

For example, instead of only having individual song plays, I can calculate:

- total plays
- unique tracks
- unique artists
- skip rate
- repeat rate
- discovery rate
- listening time
- monthly listening behavior

In [ ]:
import pandas as pd
from pathlib import Path


## 1. Load the cleaned data

In [ ]:
project = Path.cwd()
if project.name == 'notebooks':
    project = project.parent

processed = project / 'data' / 'processed'

df = pd.read_csv(processed / 'spotify_clean_history.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print('Rows:', len(df))
df.head()


## 2. Create a skip column

The Spotify `skipped` column tells us whether a listening event was skipped.

I will convert it to True/False so it is easier to use.

In [ ]:
df['is_skip'] = df['skipped'].fillna(False).astype(bool)

print(df['is_skip'].value_counts())


## 3. Track-level features

Now I want to know what happened for each song.

For every track I can calculate how many times it was played, how much time was spent on it, and how often it was skipped.

In [ ]:
track_features = df.groupby(
    ['master_metadata_track_name', 'master_metadata_album_artist_name']
).agg(
    total_plays=('master_metadata_track_name', 'size'),
    total_minutes=('minutes_played', 'sum'),
    average_minutes=('minutes_played', 'mean'),
    skip_count=('is_skip', 'sum'),
    unique_days=('timestamp', lambda x: x.dt.date.nunique())
).reset_index()

track_features.head()


### What did `groupby()` do?

`groupby()` puts rows with the same track and artist together.

Then `agg()` calculates values for each group.

## 4. Calculate skip rate

Skip rate tells us what percentage of a track's plays were skipped.

Formula:

`skip rate = skip count / total plays`

In [ ]:
track_features['skip_rate'] = (
    track_features['skip_count'] / track_features['total_plays']
)

track_features[
    ['master_metadata_track_name', 'total_plays', 'skip_count', 'skip_rate']
].head(10)


## 5. Repeat rate and discovery rate

I want two simple measures:

- **Repeat rate:** how much of my listening comes from tracks played more than once.
- **Discovery rate:** how much of my track collection was played only once.

These are simple behavioral measures, not official Spotify definitions.

In [ ]:
repeat_rate = (track_features['total_plays'] > 1).mean()
discovery_rate = (track_features['total_plays'] == 1).mean()

print('Repeat rate:', round(repeat_rate * 100, 2), '%')
print('Discovery rate:', round(discovery_rate * 100, 2), '%')


## 6. Listening at night and on weekends

These features help describe when I listen.

For this first version, the definitions are:

- night = 10 PM to 5 AM
- weekend = Saturday or Sunday

In [ ]:
df['is_night'] = (df['hour'] >= 22) | (df['hour'] <= 5)
df['is_weekend'] = df['is_weekend'].astype(bool)

print('Night listening:', round(df['is_night'].mean() * 100, 2), '%')
print('Weekend listening:', round(df['is_weekend'].mean() * 100, 2), '%')


## 7. Create monthly features

This part is important for the clustering model later.

Instead of treating my whole listening history as one observation, I will treat **each month as one observation**.

Example:

`2021-06` = one observation

`2021-07` = one observation

`2021-08` = one observation

In [ ]:
monthly = df.groupby('month').agg(
    total_plays=('master_metadata_track_name', 'size'),
    unique_tracks=('master_metadata_track_name', 'nunique'),
    unique_artists=('master_metadata_album_artist_name', 'nunique'),
    total_minutes=('minutes_played', 'sum'),
    average_minutes=('minutes_played', 'mean'),
    skip_rate=('is_skip', 'mean'),
    night_listening_ratio=('is_night', 'mean'),
    weekend_listening_ratio=('is_weekend', 'mean')
).reset_index()

monthly.head()


## 8. Add monthly repeat and discovery rates

For each month I will calculate:

- how many different tracks were repeated
- how many tracks were played only once

In [ ]:
monthly_repeat = []
monthly_discovery = []

for month, group in df.groupby('month'):
    counts = group['master_metadata_track_name'].value_counts()

    monthly_repeat.append({
        'month': month,
        'repeat_rate': (counts > 1).mean(),
        'discovery_rate': (counts == 1).mean()
    })

repeat_discovery = pd.DataFrame(monthly_repeat)

monthly = monthly.merge(
    repeat_discovery,
    on='month',
    how='left'
)

monthly.head()


## 9. Quick check of the monthly data

Each row should now represent one month.

In [ ]:
print('Number of months:', len(monthly))
monthly


## 10. Create a simple user summary

This gives an overall picture of the listening history.

In [ ]:
user_features = pd.DataFrame([{
    'total_plays': len(df),
    'unique_tracks': df['master_metadata_track_name'].nunique(),
    'unique_artists': df['master_metadata_album_artist_name'].nunique(),
    'total_minutes': df['minutes_played'].sum(),
    'average_minutes': df['minutes_played'].mean(),
    'skip_rate': df['is_skip'].mean(),
    'night_listening_ratio': df['is_night'].mean(),
    'weekend_listening_ratio': df['is_weekend'].mean(),
    'repeat_rate': repeat_rate,
    'discovery_rate': discovery_rate
}])

user_features.T


## 11. Save the feature files

These files will be used by the next notebooks.

In [ ]:
track_features.to_csv(
    processed / 'spotify_track_behavior_features.csv',
    index=False
)

monthly.to_csv(
    processed / 'spotify_monthly_behavior_features.csv',
    index=False
)

user_features.to_csv(
    processed / 'spotify_user_behavior_features.csv',
    index=False
)

print('Feature files saved.')


## Done

The cleaned listening events have now been converted into useful behavioral features.

Next: EDA — looking at the patterns in these features.